In [1]:
import numpy as np

def get_hms(sec12):
    """
    Convert seconds past 12:00 to hours, minutes, and seconds.
    """
    hrpart = sec12 / 3600
    minpart = (sec12 % 3600) / 60
    secpart = sec12 % 60
    return hrpart, minpart, secpart

def get_errfunc(sec12):
    """
    Calculate the error function for a given time in seconds past 12:00.
    """
    # Convert seconds to hours, minutes, and seconds
    hrpart, minpart, secpart = get_hms(sec12)

    # Calculate the angles of the hour, minute, and second hands
    hrhandangle = hrpart * 30
    minhandangle = minpart * 6
    sechandangle = secpart * 6
    
    # Calculate the angles between the hands and find the two smallest angles
    between_hand_angles = sorted([min((a1:=abs(hrhandangle - minhandangle)), 360-a1),
                                  min((a2:=abs(hrhandangle - sechandangle)), 360-a2),
                                  min((a3:=abs(minhandangle - sechandangle)), 360-a3)])[:2]
    
    # Calculate the error function as the sum of squared deviations from 90 degrees
    return sum([(i-90)**2 for i in between_hand_angles])

# Calculate the angles of the hour, minute, and second hands for each second in a 12-hour period
sec_of_12hrs = np.arange(12*3600)

# Calculate the "error" function, based on how close the angles are to 90 degrees
errfunc = np.vectorize(get_errfunc)(sec_of_12hrs)

# Use the lowest local minima of the error function to identify likely ranges for a global minimum
# (i.e. a time when the hands are closest to being at right angles)
minpts = (z:=np.column_stack((errfunc[:-2],errfunc[1:-1],errfunc[2:])))[(f:=(z[:,1]<z[:,0]) & (z[:,1]<z[:,2]))]
minpts_ix = np.arange(1, 12*3600-1)[f]
f = minpts[:,1] < np.min(np.max(minpts[:,::2], axis=1))
minpts_ix,minpts = [i[f] for i in (minpts_ix, minpts)]

# Extract the times corresponding to the identified minima
s12lims = np.expand_dims(sec_of_12hrs[minpts_ix], 1) + np.array([-1.,0.,1.])
errlims = np.vectorize(get_errfunc)(s12lims)

# Binary search to find the minimum error function value, and corresponding time, to high precision
while np.max(np.ptp(errlims[:,::2], axis=1)) > 1e-10:
    s12lims = (((z:=np.repeat(s12lims, 2, axis=1))[:,:-1] + z[:,1:]) / 2)
    errlims = np.vectorize(get_errfunc)(s12lims)

    rix = np.expand_dims(np.arange(errlims.shape[0]), 1)
    cix = np.argmin(errlims, axis=1, keepdims=True) + [[-1,0,1]]
    s12lims,errlims = [i[rix,cix] for i in (s12lims, errlims)]
    
# Extract the times corresponding to the identified minima
closest_times = s12lims[:,1][(z:=np.round(errlims[:,1],10)) == np.min(z)]

# Print the times when the hands are closest to being at right angles
np.column_stack(np.vectorize(get_hms)(closest_times))

array([[ 3.8177996 , 49.06797613,  4.07856798],
       [ 8.1822004 , 10.93202387, 55.92143202]])